# Module 17 - Retrieval-augmented generation

Use this notebook after `tests/test_rag.py` is passing. The notebook builds a small RAG pipeline in layers: chunks, embeddings, vector search, prompt assembly, retrieval, then generation through a backend.

The deliverable is the RAG postmortem: what you indexed, what retrieved well, where retrieval failed, and what you would improve before using this as part of the assistant.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from g2c.inference import Backend, BackendInfo, InferenceResult, load_selected_backend
from g2c.notebook_extras.sampling import printable
from g2c.rag import (
    DEFAULT_INSTRUCTION,
    DEFAULT_OLLAMA_EMBED_MODEL,
    DEFAULT_SYSTEM,
    Chunk,
    DenseRetriever,
    HashEmbedder,
    NumpyVectorStore,
    OllamaEmbedder,
    RAGPipeline,
    RetrievedChunk,
    assemble_rag_prompt,
    chunk_text,
    cosine_similarity,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the RAG tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 17 TODOs in `g2c/rag/`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_rag.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 17 RAG tests are not passing yet."

## Display helpers

In [ ]:
def short(text: Any, limit: int = 220) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def show_chunks(chunks: list[Chunk], *, limit: int = 8) -> None:
    rows = []
    for i, chunk in enumerate(chunks[:limit], start=1):
        rows.append(
            {
                "i": i,
                "source": chunk.source,
                "span": f"{chunk.start}:{chunk.end}",
                "chars": len(chunk.text),
                "text": short(chunk.text, 120),
            }
        )
    display(Markdown(markdown_table(rows, ["i", "source", "span", "chars", "text"])))
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")


def show_retrieved(results: list[RetrievedChunk]) -> None:
    rows = []
    for r in results:
        rows.append(
            {
                "rank": r.rank,
                "score": f"{r.score:.3f}",
                "source": r.chunk.source,
                "text": short(r.chunk.text, 180),
            }
        )
    display(Markdown(markdown_table(rows, ["rank", "score", "source", "text"])))


def show_rag_answer(answer) -> None:
    print(printable(answer.answer))
    print()
    print("Sources:")
    show_retrieved(answer.retrieved)
    print("metadata:", answer.metadata)


## A tiny corpus

The first pass uses a deliberately small in-memory corpus so the moving parts are visible. Later cells swap in course docs and optional Ollama embeddings.

In [ ]:
documents = {
    "cities.md": (
        "Madrid is the capital of Spain. It is the largest city in Spain and home to many museums.\n\n"
        "Paris is the capital of France. The Seine river runs through the city.\n\n"
        "Tokyo is the capital of Japan and one of the largest metropolitan areas in the world."
    ),
    "fruit.md": (
        "Bananas are yellow fruit rich in potassium.\n\n"
        "Apples grow on trees and can be red, green, or yellow.\n\n"
        "Oranges are citrus fruit and are often used for juice."
    ),
    "course.md": (
        "Module 16 introduces inference backends and ProdLM.\n\n"
        "Module 17 introduces retrieval augmented generation over an external corpus.\n\n"
        "Module 18 adds tools so the assistant can act outside text generation."
    ),
}

for source, text in documents.items():
    print(source, len(text), "chars")


## Exercise 1 - Chunk documents

`chunk_text` turns each source document into overlapping retrievable slices. The key parameters are `chunk_size` and `chunk_overlap`.

In [ ]:
all_chunks: list[Chunk] = []
for source, text in documents.items():
    all_chunks.extend(
        chunk_text(
            text,
            source=source,
            chunk_size=120,
            chunk_overlap=30,
            metadata={"corpus": "toy"},
        )
    )

print("chunks:", len(all_chunks))
show_chunks(all_chunks)

In [ ]:
# Inspect the overlap directly on the first document.
city_chunks = [chunk for chunk in all_chunks if chunk.source == "cities.md"]
show_chunks(city_chunks)

if len(city_chunks) >= 2:
    previous = city_chunks[0]
    current = city_chunks[1]
    overlap = previous.end - current.start
    print("actual overlap:", overlap)
    print("previous suffix:", repr(previous.text[-overlap:]))
    print("current prefix:", repr(current.text[:overlap]))

Try a few chunk sizes and watch the number of chunks change. Smaller chunks retrieve more specifically; larger chunks carry more context but blur the embedding.

In [ ]:
chunk_size_sweep = []
for size in [60, 100, 160, 240]:
    overlap = max(0, size // 5)
    n_chunks = sum(
        len(chunk_text(text, source=source, chunk_size=size, chunk_overlap=overlap))
        for source, text in documents.items()
    )
    chunk_size_sweep.append({"chunk_size": size, "overlap": overlap, "chunks": n_chunks})

display(Markdown(markdown_table(chunk_size_sweep, ["chunk_size", "overlap", "chunks"])))

## Exercise 2 - Hash embeddings

`HashEmbedder` is a toy lexical embedder. It is not semantic, but it makes the same vector-store math work without an external embedding model.

In [ ]:
hash_embedder = HashEmbedder(dim=512, ngram_range=(3, 5), seed=0)
chunk_vectors = hash_embedder.embed([chunk.text for chunk in all_chunks])

print("vectors:", chunk_vectors.shape, chunk_vectors.dtype)
print("first five row norms:", np.linalg.norm(chunk_vectors[:5], axis=1))

In [ ]:
probe_texts = [
    "Madrid is the capital of Spain.",
    "Madrid is Spain's capital city.",
    "What is the capital of Spain?",
    "Bananas are a yellow fruit.",
]
probe_vectors = hash_embedder.embed(probe_texts)

rows = []
for i, a in enumerate(probe_texts):
    for j, b in enumerate(probe_texts):
        if j <= i:
            continue
        rows.append(
            {
                "a": short(a, 45),
                "b": short(b, 45),
                "cosine": f"{cosine_similarity(probe_vectors[i], probe_vectors[j]):.3f}",
            }
        )

display(Markdown(markdown_table(rows, ["a", "b", "cosine"])))

## Exercise 3 - Build and search a vector store

A flat vector store keeps chunks and vectors in matching order. Search is one dot product per chunk plus a top-k selection.

In [ ]:
store = NumpyVectorStore(dim=hash_embedder.dim)
store.add(all_chunks, chunk_vectors)
print(store)

query = "What city is the capital of Spain?"
query_vector = hash_embedder.embed([query])[0]
raw_results = store.search(query_vector, k=4)

rows = []
for rank, (chunk, score) in enumerate(raw_results, start=1):
    rows.append(
        {
            "rank": rank,
            "score": f"{score:.3f}",
            "source": chunk.source,
            "text": short(chunk.text, 180),
        }
    )

display(Markdown(markdown_table(rows, ["rank", "score", "source", "text"])))

In [ ]:
query_examples = [
    "What city is the capital of Spain?",
    "Which module adds tools?",
    "What fruit has potassium?",
    "Which river runs through Paris?",
]

for query in query_examples:
    qvec = hash_embedder.embed([query])[0]
    results = store.search(qvec, k=2)
    print("=" * 80)
    print(query)
    for rank, (chunk, score) in enumerate(results, start=1):
        print(f"[{rank}] score={score:.3f} source={chunk.source} text={short(chunk.text, 140)}")

## Exercise 4 - Use the retriever abstraction

`DenseRetriever` wires the embedder and vector store into the interface later assistant modules expect: `retrieve(query, k)` returns ranked chunks.

In [ ]:
retriever = DenseRetriever(hash_embedder, store)
retrieved = retriever.retrieve("Which module adds tools to the assistant?", k=3)
show_retrieved(retrieved)

## Exercise 5 - Assemble a RAG prompt

Prompt assembly is the augmentation step. The retrieved chunks are numbered so the model can cite them.

In [ ]:
question = "Which module adds tools to the assistant?"
prompt = assemble_rag_prompt(question, retrieved)
print(prompt.text)

In [ ]:
empty_prompt = assemble_rag_prompt("What is the population of Pluto?", [])
print(empty_prompt.text)

## Exercise 6 - End-to-end RAG with a fake backend

Before calling a live model, use a deterministic backend. This confirms retrieval and prompt assembly are wired correctly.

In [ ]:
class FakeBackend(Backend):
    def __init__(self, scripted_answer: str = "MOCK ANSWER") -> None:
        self.scripted_answer = scripted_answer
        self.last_prompt: str | None = None
        self.last_kwargs: dict[str, Any] | None = None
        self._info = BackendInfo(name="fake", model_id="fake-rag")

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        self.last_prompt = prompt
        self.last_kwargs = {
            "max_new_tokens": max_new_tokens,
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p,
        }
        return InferenceResult(
            prompt=prompt,
            completion=self.scripted_answer,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(self.scripted_answer.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [ ]:
fake_backend = FakeBackend("Module 18 adds tools to the assistant [1].")
fake_pipeline = RAGPipeline(retriever, fake_backend)
fake_answer = fake_pipeline.answer("Which module adds tools to the assistant?", k=3)
show_rag_answer(fake_answer)

print("\nPrompt sent to backend:")
print(fake_backend.last_prompt)


## Exercise 7 - Index course module docs

The toy corpus is enough to understand the mechanics. A useful RAG pipeline needs a corpus you care about. This cell indexes the course module markdown files with the hash embedder.

In [ ]:
DOC_GLOB = "*.md"
DOC_LIMIT = 24
COURSE_CHUNK_SIZE = 1400
COURSE_CHUNK_OVERLAP = 200

module_dir = repo_root / "docs" / "modules"
module_paths = sorted(module_dir.glob(DOC_GLOB))[:DOC_LIMIT]
print("documents:", len(module_paths))
for path in module_paths[:8]:
    print(" ", path.relative_to(repo_root))

course_chunks: list[Chunk] = []
start = time.perf_counter()
for path in module_paths:
    text = path.read_text(encoding="utf-8")
    course_chunks.extend(
        chunk_text(
            text,
            source=str(path.relative_to(repo_root)),
            chunk_size=COURSE_CHUNK_SIZE,
            chunk_overlap=COURSE_CHUNK_OVERLAP,
            metadata={"kind": "course-doc"},
        )
    )
elapsed = time.perf_counter() - start
print(f"chunks: {len(course_chunks):,} built in {elapsed:.2f}s")
show_chunks(course_chunks, limit=6)

In [ ]:
course_embedder = HashEmbedder(dim=1024, ngram_range=(3, 5), seed=17)
start = time.perf_counter()
course_vectors = course_embedder.embed([chunk.text for chunk in course_chunks])
course_embed_seconds = time.perf_counter() - start

course_store = NumpyVectorStore(dim=course_embedder.dim)
course_store.add(course_chunks, course_vectors)
course_retriever = DenseRetriever(course_embedder, course_store)

print(course_store)
print(f"hash embedding wall time: {course_embed_seconds:.2f}s")

In [ ]:
course_questions = [
    "Which module introduces retrieval augmented generation?",
    "What does Module 16 build?",
    "Which module adds tools?",
    "What is the deliverable for the capstone?",
]

for question in course_questions:
    print("=" * 80)
    print(question)
    show_retrieved(course_retriever.retrieve(question, k=3))

## Exercise 8 - Optional semantic embeddings with Ollama

Hash embeddings are lexical. If you ran `./prodlm.sh`, you should also have `nomic-embed-text` available through Ollama, and the next cell will build a semantic index over the same chunks. Skip it if you do not have Ollama running with `nomic-embed-text`.

In [ ]:
OLLAMA_EMBED_MODEL = DEFAULT_OLLAMA_EMBED_MODEL
OLLAMA_EMBED_DIM = 768
OLLAMA_DOC_LIMIT = 12

ollama_retriever = None
ollama_embedder = OllamaEmbedder(OLLAMA_EMBED_MODEL, dim=OLLAMA_EMBED_DIM)
ollama_chunks = course_chunks[:OLLAMA_DOC_LIMIT]
start = time.perf_counter()
ollama_vectors = ollama_embedder.embed([chunk.text for chunk in ollama_chunks])
elapsed = time.perf_counter() - start
ollama_store = NumpyVectorStore(dim=ollama_embedder.dim)
ollama_store.add(ollama_chunks, ollama_vectors)
ollama_retriever = DenseRetriever(ollama_embedder, ollama_store)
print(f"embedded {len(ollama_chunks)} chunks in {elapsed:.2f}s")

In [ ]:
semantic_question = "What part of the course lets the assistant use information outside its weights?"

print("Hash retriever:")
show_retrieved(course_retriever.retrieve(semantic_question, k=3))

if ollama_retriever is not None:
    print("Ollama semantic retriever:")
    show_retrieved(ollama_retriever.retrieve(semantic_question, k=3))

## Exercise 9 - Optional live RAG with model selection

The default live backend is ProdLM. To test a course-trained model instead, set `MODEL_SELECTION = "course"` for the strongest course artifact, or set it to a base artifact name like `"TinyLLM-30M"`; the loader will prefer `-DPO`, then `-SFT`, then the base artifact.

In [ ]:
MODEL_SELECTION = "ProdLM"  # "ProdLM", "course", or an artifact base/name such as "TinyLLM-30M"
PRODLM_MODEL_ID = None  # optional Ollama tag override when MODEL_SELECTION == "ProdLM"
LIVE_DEVICE = "auto"
LIVE_TORCH_DTYPE = "float16"
USE_SEMANTIC_RETRIEVER_IF_AVAILABLE = True

rag_backend = None
try:
    rag_backend = load_selected_backend(
        MODEL_SELECTION,
        repo_root=repo_root,
        prodlm_model_id=PRODLM_MODEL_ID,
        device=LIVE_DEVICE,
        torch_dtype=LIVE_TORCH_DTYPE,
        required=False,
    )
    if rag_backend is None:
        print("No live backend loaded. Run ./prodlm.sh or choose an available artifact.")
    else:
        print("loaded:", rag_backend.info)
except Exception as exc:
    print(f"Live backend unavailable: {type(exc).__name__}: {exc}")

live_retriever = ollama_retriever if (USE_SEMANTIC_RETRIEVER_IF_AVAILABLE and ollama_retriever is not None) else course_retriever
print("retriever:", live_retriever)

In [ ]:
live_pipeline = None
if rag_backend is not None:
    live_pipeline = RAGPipeline(live_retriever, rag_backend)
else:
    print("No live backend loaded.")


def ask_rag(question: str, *, k: int = 4, max_new_tokens: int = 220):
    if live_pipeline is None:
        print("No live RAG pipeline loaded.")
        return None
    try:
        answer = live_pipeline.answer(
            question,
            k=k,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
        )
    except Exception as exc:
        print(f"RAG call failed: {type(exc).__name__}: {exc}")
        return None
    show_rag_answer(answer)
    return answer

In [ ]:
live_question = "What does Module 17 teach, and why does retrieval quality matter?"
live_answer = ask_rag(live_question, k=4)

## Exercise 10 - Failure-mode probes

Good RAG work includes negative examples. You want to know when the model refuses, when retrieval misses, and when the generator invents facts anyway.

In [ ]:
failure_questions = [
    {"bucket": "answerable", "question": "Which module adds tools to the assistant?"},
    {"bucket": "paraphrase", "question": "Where in the course does the model get external memory?"},
    {"bucket": "unanswerable", "question": "What is the population of Pluto according to these course docs?"},
]

probe_rows = []
for item in failure_questions:
    retrieved = live_retriever.retrieve(item["question"], k=3)
    probe_rows.append(
        {
            "bucket": item["bucket"],
            "question": item["question"],
            "top source": retrieved[0].chunk.source if retrieved else "",
            "top score": f"{retrieved[0].score:.3f}" if retrieved else "",
            "top text": short(retrieved[0].chunk.text, 130) if retrieved else "",
        }
    )

display(Markdown(markdown_table(probe_rows, ["bucket", "question", "top source", "top score", "top text"])))

Now run the same probes through the live model. Requires a live backend; the cell prints a friendly skip message per question if no live RAG pipeline is loaded.

In [ ]:
for item in failure_questions:
    print("=" * 80)
    print(item["bucket"], "-", item["question"])
    ask_rag(item["question"], k=3, max_new_tokens=180)

## Exercise 11 - Persist an index extension

The course package keeps `NumpyVectorStore` in memory, but persisting an index is a natural extension. This cell gives a small starter shape you can adapt for the postmortem or an optional script.

In [ ]:
def save_store_snapshot(store: NumpyVectorStore, path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    np.save(path / "vectors.npy", store.vectors)
    chunks_json = [
        {
            "text": chunk.text,
            "source": chunk.source,
            "start": chunk.start,
            "end": chunk.end,
            "metadata": chunk.metadata,
        }
        for chunk in store.chunks
    ]
    (path / "chunks.json").write_text(json.dumps(chunks_json, indent=2), encoding="utf-8")


def load_store_snapshot(path: Path, *, dim: int) -> NumpyVectorStore:
    vectors = np.load(path / "vectors.npy")
    chunks_data = json.loads((path / "chunks.json").read_text(encoding="utf-8"))
    chunks = [Chunk(**item) for item in chunks_data]
    store = NumpyVectorStore(dim=dim)
    store.add(chunks, vectors)
    return store

snapshot_dir = repo_root / "data" / "module17-rag" / "hash-index-snapshot"
save_store_snapshot(course_store, snapshot_dir)
round_trip_store = load_store_snapshot(snapshot_dir, dim=course_embedder.dim)
print(round_trip_store)
print("same vectors:", np.allclose(course_store.vectors, round_trip_store.vectors))
print("same first chunk:", course_store.chunks[0] == round_trip_store.chunks[0])

## Postmortem notes

Write `docs/rag-postmortem.md` in 3-4 paragraphs. Cover:

- What you indexed: corpus, chunk size, overlap, embedder, vector count.
- What worked: question types where retrieval reliably surfaced the right chunk.
- Where it broke: chunking, embedding, retrieval, prompt, or model refusal/hallucination.
- What you would build next: hybrid retrieval, re-ranking, smarter chunking, larger embedder, or a better eval set.